# Stage 4 — Temporal Fusion Transformer

A strict, notebook-local TFT experiment for station `207241-at`. The train and sealed test feature artifacts are kept physically independent: test encoders never include train observations.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import random
from pathlib import Path

import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from lightning.pytorch import Trainer
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import TorchNormalizer
from pytorch_forecasting.metrics import QuantileLoss

from src.config import TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

SEED = 42
ENCODER_LENGTH = 168
PREDICTION_LENGTH = 24
QUANTILES = (0.10, 0.50, 0.90)
BATCH_SIZE = 64
MAX_EPOCHS = 10
PROCESSED_DIR = Path("data/processed")

CALENDAR_COLUMNS = [
    "utc_hour_sin", "utc_hour_cos",
    "utc_day_of_week_sin", "utc_day_of_week_cos",
    "utc_day_of_year_sin", "utc_day_of_year_cos",
]
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())
ENCODER_ONLY_REALS = [column for column in FEATURE_COLUMNS if column not in CALENDAR_COLUMNS]

if TARGET_STATION_ID != "207241-at":
    raise ValueError(f"This MVP is fixed to station 207241-at, got {TARGET_STATION_ID!r}")
if PREDICTION_LENGTH != len(TARGET_COLUMNS):
    raise ValueError("TFT prediction length must match the Stage-3 target horizon")
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required for this experiment and is not available; CPU fallback is disabled")

pl.seed_everything(SEED, workers=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

configuration = pd.DataFrame([
    {
        "station_id": TARGET_STATION_ID, "device": "mps", "seed": SEED,
        "encoder_hours": ENCODER_LENGTH, "forecast_hours": PREDICTION_LENGTH,
        "quantiles": QUANTILES, "batch_size": BATCH_SIZE, "epochs": MAX_EPOCHS,
        "hidden_size": 64, "lstm_layers": 2, "attention_heads": 4,
        "hidden_continuous_size": 32, "dropout": 0.1,
        "learning_rate": 0.001, "gradient_clip_val": 0.1,
        "target_normalization": "standard, fit on train station only",
        "deterministic_kernels": False,
        "known_decoder_reals": len(CALENDAR_COLUMNS),
        "encoder_only_reals": len(ENCODER_ONLY_REALS),
    }
])
display(configuration)


## Input contract and leakage boundary

Only the Stage-3 train and sealed-test feature Parquets are read. Each is processed independently, so the test encoder cannot use train observations. The input check fails clearly if an artifact, required column, target vector, timestamp, or station identity is invalid. The Boolean `imputed` flag is converted only in the notebook-local copy because TFT continuous inputs must be numeric.

In [ ]:
def read_feature_artifact(path: Path, *, station_id: str, artifact_name: str) -> pd.DataFrame:
    """Load one Stage-3 artifact and enforce the TFT input contract."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing feature artifact for {station_id}: {path}")
    frame = pd.read_parquet(path).copy()
    required = {"timestamp", "station_id", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"{station_id} {artifact_name} artifact is missing required columns: {missing}")
    if frame.empty:
        raise ValueError(f"{station_id} {artifact_name} artifact is empty")
    if frame["station_id"].dropna().unique().tolist() != [station_id]:
        raise ValueError(f"{station_id} {artifact_name} artifact contains another station")
    frame = frame.sort_values("timestamp").reset_index(drop=True)
    if frame["timestamp"].duplicated().any():
        raise ValueError(f"{station_id} {artifact_name} artifact has duplicate timestamps")
    if not pd.api.types.is_datetime64tz_dtype(frame["timestamp"]):
        raise ValueError(f"{station_id} {artifact_name} timestamps must be timezone-aware")
    target_rows = frame["target_valid"].eq(True)
    if frame.loc[target_rows, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(f"{station_id} {artifact_name} has incomplete target vectors in target-valid rows")
    frame["imputed"] = frame["imputed"].astype(float)
    return frame


def build_complete_segments(frame: pd.DataFrame, *, artifact_name: str) -> pd.DataFrame:
    """Keep only contiguous rows with every Stage-3 predictor present."""
    complete = frame[FEATURE_COLUMNS].notna().all(axis=1)
    contiguous = frame["timestamp"].diff().eq(pd.Timedelta(hours=1))
    starts_segment = complete & (~complete.shift(fill_value=False) | ~contiguous)
    segment_number = starts_segment.cumsum()
    result = frame.loc[complete].copy()
    # Segment labels are partition-local, stable identifiers rather than features.
    # This lets the sealed test reuse fitted categorical encoders without joining it to train.
    result["segment_id"] = "segment_" + segment_number.loc[complete].astype(str).str.zfill(4)
    result["time_idx"] = ((result["timestamp"] - pd.Timestamp("1970-01-01", tz="UTC")) / pd.Timedelta(hours=1)).astype(int)
    # MPS does not support float64 tensors; this copy is notebook-local.
    result[FEATURE_COLUMNS] = result[FEATURE_COLUMNS].astype(np.float32)
    if result.empty:
        raise ValueError(f"{artifact_name} artifact has no complete feature rows")
    return result.reset_index(drop=True)


station_id = TARGET_STATION_ID
train_features = read_feature_artifact(
    PROCESSED_DIR / f"{station_id}_train_features.parquet",
    station_id=station_id, artifact_name="train",
)
test_features = read_feature_artifact(
    PROCESSED_DIR / f"{station_id}_test_features.parquet",
    station_id=station_id, artifact_name="test",
)
train_data = build_complete_segments(train_features, artifact_name="train")
test_data = build_complete_segments(test_features, artifact_name="test")

if train_data["timestamp"].max() >= test_data["timestamp"].min():
    raise ValueError("Train and sealed test artifacts overlap or are not chronological")


## Fixed-length, complete sequences

Rows with any missing Stage-3 predictor are excluded and the remaining rows are split at every discontinuity. TFT samples use exactly 168 encoder hours followed by 24 decoder hours. After candidate construction, decoded-index filtering retains only forecasts whose issue timestamp has a complete, observed 24-hour target vector (`target_valid`). Short or incomplete segments are intentionally absent from the strict cohort.

In [ ]:
def make_dataset(data: pd.DataFrame, *, template: TimeSeriesDataSet | None = None) -> TimeSeriesDataSet:
    """Construct fixed-length TFT samples without allowing missing timesteps."""
    if template is not None:
        return TimeSeriesDataSet.from_dataset(
            template, data, stop_randomization=True, predict=False
        )
    return TimeSeriesDataSet(
        data,
        time_idx="time_idx",
        target="water_level",
        group_ids=["segment_id"],
        min_encoder_length=ENCODER_LENGTH,
        max_encoder_length=ENCODER_LENGTH,
        min_prediction_length=PREDICTION_LENGTH,
        max_prediction_length=PREDICTION_LENGTH,
        static_categoricals=["station_id"],
        time_varying_known_reals=CALENDAR_COLUMNS,
        time_varying_unknown_reals=ENCODER_ONLY_REALS,
        target_normalizer=TorchNormalizer(method="standard"),
        allow_missing_timesteps=False,
        randomize_length=False,
    )


def filter_target_valid_sequences(
    dataset: TimeSeriesDataSet, data: pd.DataFrame, *, artifact_name: str
) -> TimeSeriesDataSet:
    """Keep samples whose decoder starts immediately after a valid issue time."""
    eligibility = data.set_index(["segment_id", "time_idx"])["target_valid"]

    def issue_is_eligible(index: pd.DataFrame) -> pd.Series:
        issue_keys = list(zip(index["segment_id"], index["time_idx_first_prediction"] - 1))
        return pd.Series(
            [bool(eligibility.get(key, False)) for key in issue_keys], index=index.index
        )

    filtered = dataset.filter(issue_is_eligible, copy=True)
    if len(filtered) == 0:
        raise ValueError(
            f"{artifact_name} artifact has no fixed {ENCODER_LENGTH}-to-{PREDICTION_LENGTH} target-valid sequences"
        )
    return filtered


train_candidates = make_dataset(train_data)
test_candidates = make_dataset(test_data, template=train_candidates)
train_dataset = filter_target_valid_sequences(train_candidates, train_data, artifact_name="train")
test_dataset = filter_target_valid_sequences(test_candidates, test_data, artifact_name="test")

candidate_sequence_counts = pd.DataFrame([
    {"artifact": "train", "complete_rows": len(train_data), "candidate_sequences": len(train_candidates), "scored_sequences": len(train_dataset)},
    {"artifact": "test", "complete_rows": len(test_data), "candidate_sequences": len(test_candidates), "scored_sequences": len(test_dataset)},
])
display(candidate_sequence_counts)


## TFT training

The six calendar signals are known to the decoder. Water level, imputation, weather, lags, and rolling features are encoder-only. Feature and target normalizers are fitted on the train station and reused unchanged for test. Training is deliberately fixed: a medium TFT, Adam at 0.001, 0.1 gradient clipping, batch size 64, and 50 epochs on MPS. Logging, checkpoints, validation, early stopping, and learning-rate adaptation are disabled.

In [ ]:
train_loader = train_dataset.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
test_loader = test_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

tft = TemporalFusionTransformer.from_dataset(
    train_dataset,
    hidden_size=64,
    lstm_layers=2,
    attention_head_size=4,
    hidden_continuous_size=32,
    dropout=0.1,
    learning_rate=0.001,
    loss=QuantileLoss(quantiles=list(QUANTILES)),
    output_size=len(QUANTILES),
    optimizer="Adam",
    reduce_on_plateau_patience=0,
    log_interval=-1,
    logging_metrics=[],
)
parameter_count = sum(parameter.numel() for parameter in tft.parameters() if parameter.requires_grad)
display(pd.DataFrame([{"station_id": station_id, "device": "mps", "trainable_parameters": parameter_count}]))

trainer = Trainer(
    accelerator="mps",
    devices=1,
    max_epochs=MAX_EPOCHS,
    gradient_clip_val=0.1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)
trainer.fit(tft, train_dataloaders=train_loader)


## Sealed-test quantile forecasts

Predictions are generated only for the filtered sealed-test sequences. The model returns p10, p50, and p90 at each of the 24 horizons; the following checks verify the three-quantile and timestamp alignment before any score is calculated.

In [ ]:
prediction_result = tft.predict(
    test_loader,
    mode="quantiles",
    return_x=True,
    trainer_kwargs={
        "accelerator": "mps", "devices": 1, "logger": False,
        "enable_checkpointing": False, "enable_model_summary": False,
    },
)
quantile_predictions = prediction_result.output.detach().cpu().numpy()
decoder_time_idx = prediction_result.x["decoder_time_idx"].detach().cpu().numpy()
if quantile_predictions.shape[1:] != (PREDICTION_LENGTH, len(QUANTILES)):
    raise ValueError(f"Unexpected TFT prediction shape: {quantile_predictions.shape}")
if decoder_time_idx.shape != quantile_predictions.shape[:2]:
    raise ValueError("TFT decoder timestamps do not align with predictions")

timestamp_lookup = test_data.drop_duplicates("time_idx").set_index("time_idx")["timestamp"]
actual_lookup = test_data.drop_duplicates("time_idx").set_index("time_idx")["water_level"]
actual_values = actual_lookup.reindex(decoder_time_idx.ravel()).to_numpy().reshape(decoder_time_idx.shape)
target_timestamps = timestamp_lookup.reindex(decoder_time_idx.ravel()).to_numpy().reshape(decoder_time_idx.shape)
issue_timestamps = timestamp_lookup.reindex((decoder_time_idx[:, 0] - 1)).to_numpy()
if np.isnan(actual_values).any() or pd.isna(target_timestamps).any() or pd.isna(issue_timestamps).any():
    raise ValueError("Filtered sealed test sequences contain incomplete target timestamps or values")

scored_timestamp_ranges = pd.DataFrame([{
    "station_id": station_id,
    "issue_start": issue_timestamps.min(), "issue_end": issue_timestamps.max(),
    "target_start": target_timestamps.min(), "target_end": target_timestamps.max(),
    "forecast_origins": len(issue_timestamps), "scored_values": actual_values.size,
}])
display(scored_timestamp_ranges)


## Accuracy and calibration

The aggregate and horizon-specific tables report median MAE/RMSE, pinball loss at each requested quantile, empirical p10–p90 coverage, interval width, and the rate at which quantiles cross. All values are calculated from the same filtered sealed-test cohort shown above.

In [ ]:
def pinball_loss(actual: np.ndarray, prediction: np.ndarray, quantile: float) -> float:
    error = actual - prediction
    return float(np.maximum(quantile * error, (quantile - 1.0) * error).mean())


def metric_row(actual: np.ndarray, prediction: np.ndarray) -> dict[str, float]:
    p10, p50, p90 = (prediction[..., index] for index in range(len(QUANTILES)))
    crossing = (p10 > p50) | (p50 > p90)
    return {
        "median_mae": float(np.abs(actual - p50).mean()),
        "median_rmse": float(np.sqrt(np.mean((actual - p50) ** 2))),
        "pinball_p10": pinball_loss(actual, p10, 0.10),
        "pinball_p50": pinball_loss(actual, p50, 0.50),
        "pinball_p90": pinball_loss(actual, p90, 0.90),
        "empirical_80pct_coverage": float(((actual >= p10) & (actual <= p90)).mean()),
        "mean_interval_width": float((p90 - p10).mean()),
        "quantile_crossing_rate": float(crossing.mean()),
    }


aggregate_metrics = pd.DataFrame([{
    "station_id": station_id, "forecast_origins": len(actual_values),
    "scored_values": actual_values.size, **metric_row(actual_values, quantile_predictions),
}])
per_horizon_metrics = pd.DataFrame([
    {"station_id": station_id, "horizon_hours": horizon, **metric_row(actual_values[:, horizon - 1], quantile_predictions[:, horizon - 1, :])}
    for horizon in range(1, PREDICTION_LENGTH + 1)
])
display(aggregate_metrics)
display(per_horizon_metrics)


## Forecast inspection and model interpretation

The preview gives the first five forecast origins in long form, making each target timestamp and horizon explicit. The final figures aggregate TFT encoder variable-selection weights and attention over the sealed-test forecasts. Nothing is written to disk.

In [ ]:
preview_origins = min(5, len(issue_timestamps))
forecast_preview = pd.DataFrame({
    "issue_timestamp": np.repeat(issue_timestamps[:preview_origins], PREDICTION_LENGTH),
    "target_timestamp": target_timestamps[:preview_origins].ravel(),
    "horizon_hours": np.tile(np.arange(1, PREDICTION_LENGTH + 1), preview_origins),
    "actual_water_level": actual_values[:preview_origins].ravel(),
    "p10": quantile_predictions[:preview_origins, :, 0].ravel(),
    "p50": quantile_predictions[:preview_origins, :, 1].ravel(),
    "p90": quantile_predictions[:preview_origins, :, 2].ravel(),
})
display(forecast_preview)

raw_prediction_result = tft.predict(
    test_loader, mode="raw", return_x=True,
    trainer_kwargs={
        "accelerator": "mps", "devices": 1, "logger": False,
        "enable_checkpointing": False, "enable_model_summary": False,
    },
)
interpretation = tft.interpret_output(raw_prediction_result.output, reduction="sum")
interpretation_figures = tft.plot_interpretation(interpretation)
for name, figure in interpretation_figures.items():
    figure.suptitle(name.replace("_", " "))
    display(figure)
